# Notebook 1: Einführung und Daten

**Statistik für Data Science** · Kurs 3,906 · Bachelor Computer Science · HS2026
Universität St. Gallen · Begleitmaterial zur Übung

---

Dieses Notebook führt die Inhalte von **VL01 (Einführung & Daten)** in Python aus. Die Vorlesung
zeigt die Konzepte, hier laufen sie. Der Aufbau folgt exakt den vier Abschnitten des Foliensatzes:

| Abschnitt | Thema |
|---|---|
| 1 | Datentypen und Messniveaus |
| 2 | Bias in Daten |
| 3 | Missing Values |
| 4 | Erste EDA |

**Wie dieses Notebook gedacht ist.** Es ist eine durchgerechnete Referenz, kein Aufgabenblatt.
Alle Zellen sind ausgeführt, die Ausgaben stehen darunter. Beim Projekt schlagt ihr hier nach:
jedes Muster ist so geschrieben, dass es sich auf einen beliebigen eigenen Datensatz übertragen
lässt. Abschnitt 5 fasst die Bausteine als wiederverwendbare Funktionen zusammen.

**Datensatz.** Durchgehend der Titanic-Datensatz aus `seaborn`, derselbe wie in der Vorlesung.
Für Abschnitt 3 kommt zusätzlich ein selbst erzeugter Datensatz dazu, weil sich die drei
Missing-Mechanismen an echten Daten grundsätzlich nicht zeigen lassen. Warum, steht dort.

## 0. Setup

Ein Block für alles, was das ganze Notebook braucht. Bewusst schlank gehalten: `pandas`, `numpy`,
`seaborn` und `matplotlib` reichen für VL01 vollständig aus.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Ein zentraler Seed für alles Zufällige in diesem Notebook.
# Der Syllabus verlangt, dass Notebooks Umgebung, Pakete und Seeds dokumentieren.
SEED = 42
RNG = np.random.default_rng(SEED)

# Plot-Defauls
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7.5, 4.2)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

Reproduzierbarkeit heisst, dass eine andere Person zu denselben Zahlen kommt. Dazu gehört die
Version jedes Pakets, das Ergebnisse beeinflusst.

In [ ]:
import sys

print("Python  ", sys.version.split()[0])
for modul in (np, pd, sns):
    print(f"{modul.__name__:8s}", modul.__version__)
print("Seed    ", SEED)

### Daten laden

In [ ]:
titanic = sns.load_dataset("titanic")

print(f"Zeilen: {titanic.shape[0]}, Spalten: {titanic.shape[1]}")

---

## 1. Datentypen und Messniveaus

> **Grundsatz der Vorlesung:** Die Methode hängt vom Datentyp ab. Das Messniveau
> entscheidet, welche Kennzahl überhaupt zulässig ist.

Der erste Blick auf einen unbekannten Datensatz geht immer über `.info()`.

In [ ]:
titanic.info()

Zwei Dinge stehen hier schon drin:

1. **Nicht jede Spalte ist vollständig.** `age` hat 714 von 891 Einträgen, `deck` nur 203. Das ist
   Abschnitt 3.
2. **Der dtype ist nicht das Messniveau.** Das ist die Kernaussage der Vorlesung.

Zum zweiten Punkt: `pclass` steht als `int64` in der Tabelle. Das ist der *Speichertyp*. Inhaltlich
ist `pclass` eine **Rangfolge** (1. Klasse besser als 3. Klasse), also ordinal. pandas weiss davon
nichts und würde bereitwillig einen Mittelwert ausrechnen.

> **Hinweis zur pandas-Version.** Die Vorlesung zeigt eine ältere Ausgabeform (`survived  891 non-null
> int64`). Aktuelle pandas-Versionen drucken stattdessen eine Tabelle mit Spaltennummern, und ab
> pandas 3 heisst der Texttyp `str` statt `object`. Inhaltlich ist es dasselbe.

Bevor es an die einzelnen Skalen geht, ein Überblick, der bei jedem fremden Datensatz der erste
Schritt ist: Was steht pro Spalte drin, wie viele verschiedene Werte gibt es, und wie sehen sie aus?

In [ ]:
def spalten_ueberblick(df: pd.DataFrame, beispiele: int = 3) -> pd.DataFrame:
    """Kompakter Steckbrief je Spalte: dtype, Anzahl eindeutiger Werte, fehlende Werte, Beispiele.

    Erster Schritt bei jedem unbekannten Datensatz.
    """
    zeilen = []
    for spalte in df.columns:
        s = df[spalte]
        werte = s.dropna().unique()[:beispiele]
        zeilen.append(
            {
                "dtype": str(s.dtype),
                "eindeutig": s.nunique(dropna=True),
                "fehlend": int(s.isna().sum()),
                "fehlend_%": round(s.isna().mean() * 100, 1),
                "beispiele": ", ".join(map(str, werte)),
            }
        )
    return pd.DataFrame(zeilen, index=df.columns)


spalten_ueberblick(titanic)

### Die vier Messniveaus

Die Vorlesung stellt die vier Skalen gegenüber. Kurzfassung, mit dem, was jeweils erlaubt ist:

| Skala | Eigenschaft | Erlaubte Statistik |
|---|---|---|
| **Nominal** | Kategorien ohne Ordnung | Modus, Häufigkeiten |
| **Ordinal** | Rangfolge, Abstände undefiniert | zusätzlich Median, Perzentile |
| **Intervall** | gleiche Abstände, kein natürlicher Nullpunkt | zusätzlich Mittelwert, Standardabweichung |
| **Ratio** | absoluter Nullpunkt | zusätzlich Verhältnisse, geometrisches Mittel |

**Merksatz der Vorlesung:** Je höher das Messniveau, desto mehr Operationen sind erlaubt. Jedes Niveau
erbt alle Eigenschaften der darunterliegenden Skalen.

Diese Zuordnung kann kein Programm für euch treffen, sie ist eine inhaltliche Entscheidung. Genau
deshalb schreibt man sie einmal explizit hin:

In [ ]:
# Inhaltliche Zuordnung. Sie steht bewusst als Hand-Mapping da, weil sie sich
# nicht aus dem dtype ableiten laesst.
MESSNIVEAU = {
    "survived": "nominal",      # 0/1, keine Rangfolge im statistischen Sinn
    "pclass": "ordinal",        # int64, aber eine Rangfolge
    "sex": "nominal",
    "age": "ratio",             # 0 Jahre ist ein echter Nullpunkt
    "sibsp": "ratio",           # Anzahl
    "parch": "ratio",           # Anzahl
    "fare": "ratio",            # 0 CHF ist ein echter Nullpunkt
    "embarked": "nominal",
    "class": "ordinal",         # dasselbe wie pclass, als Text
    "who": "nominal",
    "adult_male": "nominal",
    "deck": "nominal",
    "embark_town": "nominal",
    "alive": "nominal",
    "alone": "nominal",
}

zuordnung = pd.DataFrame(
    {
        "dtype": titanic.dtypes.astype(str),
        "messniveau": pd.Series(MESSNIVEAU),
    }
)
zuordnung["mittelwert_zulaessig"] = zuordnung["messniveau"].isin(["intervall", "ratio"])
zuordnung

Interessant ist die Spalte `mittelwert_zulaessig` im Vergleich zum dtype: `survived` und `pclass`
sind beide `int64`, und für beide ist der Mittelwert unzulässig. `pandas` rechnet ihn trotzdem aus.

### Nominalskala

Reine Kategorisierung, keine Ordnung. Erlaubt sind **absolute und relative Häufigkeiten** und der
**Modus**, also der häufigste Wert.

In [ ]:
print("Absolute Häufigkeiten:")
print(titanic["embarked"].value_counts(dropna=False))

print("\nRelative Häufigkeiten (in %):")
print((titanic["embarked"].value_counts(normalize=True) * 100).round(1))

print("\nModus:", titanic["embarked"].mode()[0])

### Ordinalskala

Es gibt eine klare Rangfolge, aber die Abstände zwischen den Rängen sind nicht quantifizierbar.
Der Unterschied zwischen 1. und 2. Klasse ist nicht derselbe wie zwischen 2. und 3.

Erlaubt sind zusätzlich **Median** und **Perzentile**, weil beide nur die Reihenfolge brauchen.
In pandas hält man die Ordnung in einem geordneten `Categorical` fest. Das ist der Unterschied
zwischen "pandas kennt die Reihenfolge" und "pandas sortiert alphabetisch".

In [ ]:
# Reihenfolge explizit festlegen: Third < Second < First
klasse = titanic["class"].astype(
    pd.CategoricalDtype(categories=["Third", "Second", "First"], ordered=True)
)

print("Kategorien in Reihenfolge:", list(klasse.cat.categories))
print("\nHäufigkeiten (in Rangreihenfolge):")
print(klasse.value_counts().sort_index())

In [ ]:
# Median und Quartile ordinaler Daten laufen ueber die Rangposition, nicht ueber den Wert.
# pandas rechnet auf einem Categorical bewusst keinen Median, deshalb der Umweg ueber die Codes.
raenge = klasse.cat.codes

for beschriftung, q in [("25 %", 0.25), ("Median", 0.50), ("75 %", 0.75)]:
    print(f"{beschriftung:7s}", klasse.cat.categories[int(raenge.quantile(q))])

Der Median ist `Third`: mehr als die Hälfte der Passagiere reiste dritter Klasse.

**Was hier nicht steht, ist ein Mittelwert.** Genau das ist die Falle, die die
Vorlesung mit `describe()` unabsichtlich aufstellt: pandas rechnet für `pclass` bereitwillig einen
Mittelwert aus, weil dort `int64` steht.

In [ ]:
print("describe() auf pclass:")
print(titanic["pclass"].describe()[["mean", "50%"]])

print(f"\nMittelwert von pclass: {titanic['pclass'].mean():.6f}")
print("Diese Zahl ist rechenbar, aber inhaltlich unzulässig: Klasse 2.31 gibt es nicht.")
print(f"Zulässig und aussagekräftig ist der Median: {titanic['pclass'].median():.0f}")

### Intervallskala

Gleiche Abstände, aber **kein natürlicher Nullpunkt**. Differenzen sind sinnvoll, Verhältnisse nicht.

Der Titanic-Datensatz enthält keine einzige intervallskalierte Spalte. Das ist kein Zufall:
Intervallskalen sind in der Praxis selten, die beiden Klassiker sind **Temperatur in Grad Celsius**
und **Kalenderdaten**. Beide Beispiele stehen in der Vorlesung, deshalb hier an einem Mini-Datensatz.

In [ ]:
messungen = pd.DataFrame(
    {
        "erste_session": pd.to_datetime(["2026-09-15", "2026-09-22", "2026-10-06"]),
        "temperatur_c": [10.0, 20.0, 30.0],
    }
)

# Erlaubt: Differenzen.
messungen["tage_seit_start"] = (
    messungen["erste_session"] - messungen["erste_session"].min()
).dt.days
messungen["differenz_c"] = messungen["temperatur_c"].diff()

print(messungen)

print("\nErlaubt:      20 °C minus 10 °C =", messungen["temperatur_c"][1] - messungen["temperatur_c"][0], "Grad Unterschied")
print("Unzulässig:   20 °C ist NICHT doppelt so warm wie 10 °C.")

# Die Gegenprobe macht es zwingend: dieselben Temperaturen in Fahrenheit.
fahrenheit = messungen["temperatur_c"] * 9 / 5 + 32
print(f"\nVerhältnis in Celsius:    {messungen['temperatur_c'][1] / messungen['temperatur_c'][0]:.2f}")
print(f"Verhältnis in Fahrenheit: {fahrenheit[1] / fahrenheit[0]:.2f}")
print("Dasselbe Paar Temperaturen, zwei verschiedene Verhältnisse. Also ist das Verhältnis bedeutungslos.")

Das ist der ganze Punkt der Intervallskala: Weil der Nullpunkt willkürlich gesetzt ist (0 °C ist
der Gefrierpunkt von Wasser, kein Nichtvorhandensein von Wärme), ändert sich das Verhältnis mit der
Wahl der Einheit. Bei Kelvin, das einen absoluten Nullpunkt hat, passiert das nicht: Kelvin ist
ratioskaliert.

### Ratioskala

Absoluter Nullpunkt, der ein vollständiges Nichtvorhandensein des Merkmals bedeutet. Damit sind
**alle** arithmetischen Operationen erlaubt, insbesondere Verhältnisse und das geometrische Mittel.

In [ ]:
fare = titanic["fare"]

print(f"Mittelwert:          {fare.mean():.2f}")
print(f"Standardabweichung:  {fare.std():.2f}")

# Verhaeltnisse sind hier sinnvoll, weil 0 ein echter Nullpunkt ist.
mittel_first = titanic.loc[titanic["class"] == "First", "fare"].mean()
mittel_third = titanic.loc[titanic["class"] == "Third", "fare"].mean()
print(f"\nErste Klasse zahlte im Schnitt das {mittel_first / mittel_third:.1f}-fache der dritten Klasse.")
print("Dieser Satz ist nur bei einer Ratioskala zulässig.")

# Das geometrische Mittel ist nur fuer positive Ratiodaten definiert.
positiv = fare[fare > 0]
geo = np.exp(np.log(positiv).mean())
print(f"\nGeometrisches Mittel (nur fare > 0, n = {len(positiv)}): {geo:.2f}")
print(f"Arithmetisches Mittel derselben Werte:                {positiv.mean():.2f}")

Das geometrische Mittel liegt deutlich unter dem arithmetischen. Der Grund sind einzelne sehr hohe
Ticketpreise, die den arithmetischen Mittelwert nach oben ziehen. Das ist schon der Übergang zu
Abschnitt 4, wo genau diese Schiefe sichtbar wird.

> **Take-Home Datentypen**
>
> 1. **Messniveau zuerst.** Vor jeder Kennzahl klären, welche Skala vorliegt.
> 2. **Vier Skalen:** nominal, ordinal, intervall, ratio.
> 3. **Reihenfolge im Kopf behalten.** Der dtype sagt nichts über das Messniveau.

---

## 2. Bias in Daten

> **Kernaussage der Vorlesung:** Bias ist ein **gerichteter** Fehler, eine systematische Schieflage, kein
> zufälliges Rauschen.

Der Unterschied ist entscheidend und ein Prüfungsklassiker: Eine Waage, die mal 200 g zu viel und
mal 200 g zu wenig anzeigt, im Mittel aber korrekt, hat **Rauschen**. Eine Waage, die immer 200 g zu
viel anzeigt, hat **Bias**.

Daraus folgt der Satz, der in den Sprechernotizen der Vorlesung steht und der die ganze Übung trägt:
**mehr Daten helfen gegen Bias nicht.** Sie machen einen verzerrten Schätzer nur präziser verzerrt.

Bias lässt sich nicht aus den Daten herausrechnen, aber man kann ihn sichtbar machen, indem man
eine Kennzahl auf einem Ausschnitt gegen dieselbe Kennzahl auf allen Daten stellt.

### Sampling Bias

Die Stichprobe bildet die Zielpopulation nicht ab. Hier simulieren wir das nicht, wir nehmen die
Titanic-Daten und schauen uns an, was passiert, wenn man nur einen Teil des Schiffs befragt.

In [ ]:
wahrer_mittelwert = titanic["fare"].mean()

nach_klasse = titanic.groupby("class", observed=True)["fare"].agg(["size", "mean"])
nach_klasse["abweichung"] = nach_klasse["mean"] - wahrer_mittelwert

print(f"Mittlerer Ticketpreis, alle Passagiere: {wahrer_mittelwert:.2f}\n")
print(nach_klasse)

Wer nur die erste Klasse befragt, berichtet einen mittleren Ticketpreis von rund 84 statt 32. Der
Fehler beträgt mehr als das Doppelte des wahren Wertes, und er zeigt **in eine Richtung**.

Jetzt der Punkt, um den es eigentlich geht: Hilft es, mehr Leute aus der ersten Klasse zu befragen?

In [ ]:
erste_klasse = titanic.loc[titanic["class"] == "First", "fare"]
kleine_stichprobe = erste_klasse.sample(50, random_state=SEED)

print(f"Wahrer Mittelwert (alle 891 Passagiere):        {wahrer_mittelwert:.2f}")
print(f"Nur erste Klasse, n =  {len(kleine_stichprobe):3d}:                   {kleine_stichprobe.mean():.2f}")
print(f"Nur erste Klasse, n = {len(erste_klasse):3d} (alle):              {erste_klasse.mean():.2f}")
print()
print(f"Fehler bei n =  {len(kleine_stichprobe)}: {kleine_stichprobe.mean() - wahrer_mittelwert:+.2f}")
print(f"Fehler bei n = {len(erste_klasse)}: {erste_klasse.mean() - wahrer_mittelwert:+.2f}")

Nein. Die Stichprobe wird mehr als viermal so gross, und der Fehler landet bei rund **+52**. Die
kleine Stichprobe streut um diesen Wert (hier +55.56), die grosse trifft ihn exakt, weil sie die
gesamte erste Klasse umfasst. Was der Fehler **nicht** tut: gegen null gehen.

Das ist der Unterschied zwischen Bias und Varianz in drei Zahlen. Mehr Daten verkleinern die
Streuung um den verzerrten Wert. Den Abstand zum wahren Wert lassen sie unberührt, denn der steckt
nicht in der Stichprobengrösse, sondern in der Auswahlregel.

### Survivorship Bias

Man betrachtet nur die "Überlebenden" eines Selektionsprozesses. Das klassische Beispiel sind die
Bomber aus dem Zweiten Weltkrieg, die zurückkehrten. Der Titanic-Datensatz erlaubt die Rechnung
wörtlich.

In [ ]:
vergleich = pd.DataFrame(
    {
        "alle Passagiere": titanic[["fare", "age"]].mean(),
        "nur Überlebende": titanic.loc[titanic["survived"] == 1, ["fare", "age"]].mean(),
    }
)
vergleich["Differenz"] = vergleich["nur Überlebende"] - vergleich["alle Passagiere"]
print(vergleich)

print(f"\nAnteil Überlebende: {titanic['survived'].mean():.1%}")

Wer nur die Überlebenden auswertet, berichtet einen mittleren Ticketpreis von rund 48 statt 32.
Der Grund ist genau der Selektionsprozess: Wer teuer gebucht hatte, überlebte häufiger.

Übertragen auf ein Datenprodukt: Wer die Zufriedenheit nur unter den Nutzern misst, die die App noch
haben, misst die Zufriedenheit der Gebliebenen. Die informativste Gruppe ist die, die fehlt.

### Measurement Bias und Datenqualität

Systematischer Messfehler, Sensor-Drift, Proxy-Verzerrung. Oft zeigt er sich zuerst als Wert, der
nicht sein kann. Die `describe()`-Tabelle der Vorlesung zeigt für `fare` ein Minimum von 0.00 und geht nicht
darauf ein.

In [ ]:
gratis = titanic[titanic["fare"] == 0]
print(f"Passagiere mit fare == 0: {len(gratis)}")
print(gratis[["class", "sex", "age", "fare", "survived"]].head())

print("\nVerteilung über die Klassen:")
print(gratis["class"].value_counts())

Fünfzehn Tickets zu 0. Das ist kein Rundungsfehler, sondern eine inhaltliche Frage: Handelt es sich
um Freikarten für Werftpersonal, um einen Erfassungsfehler oder um einen Platzhalter, mit dem
"unbekannt" als Null kodiert wurde? Die dritte Möglichkeit ist die gefährlichste, weil dann in
Wahrheit **fehlende Werte** als Nullen im Datensatz stehen und jede Kennzahl leise nach unten
ziehen. Beantworten lässt sich das nur mit Domänenwissen, nicht mit Code.

> **Take-Home Bias**
>
> 1. **Bias ist gerichtet.** Nicht Rauschen, sondern eine Schieflage in eine Richtung.
> 2. **Menge hilft nicht.** Mehr Daten schätzen den Bias präziser, sie entfernen ihn nicht.
> 3. **Fünf Klassiker:** Sampling, Survivorship, Confirmation, Publication, Measurement.

**Checkliste für die eigenen Projektdaten.** Vier Fragen, die vor jeder Kennzahl geklärt sein sollten:

1. **Wer ist in den Daten und wer fehlt?** Wie kam eine Zeile in den Datensatz und welcher Filter lag davor?
2. **Gibt es einen Selektionsprozess?** Sind nur Erfolgreiche, Aktive oder Gebliebene erfasst?
3. **Wie wurde gemessen?** Sensor, Selbstauskunft oder Proxy, und in welche Richtung würde der Fehler zeigen?
4. **Gibt es unmögliche Werte?** Nullen, negative Alter, Platzhalter wie -1 oder 9999.

---

## 3. Missing Values

> **Kernaussage der Vorlesung:** Die entscheidende Frage ist, **warum** Werte fehlen. Die blosse Anzahl sagt
> sehr wenig.

Dieser Abschnitt hat zwei Teile. Zuerst die Diagnose an echten Daten, dann die drei Mechanismen an
einem selbst erzeugten Datensatz. Warum der zweite Teil nötig ist, steht unten.

### 3.1 Diagnose: wie viel fehlt wo?

Die Vorlesung nennt die Zahl "177 fehlende Alterswerte", rechnet sie aber nie aus. Hier ist die
Rechnung. `isna()` liefert eine Maske aus True und False, `sum()` zählt sie, `mean()` gibt direkt
den Anteil.

In [ ]:
def fehlende_werte(df: pd.DataFrame) -> pd.DataFrame:
    """Übersicht der fehlenden Werte je Spalte, absteigend sortiert.

    Erster Schritt jeder Missing-Value-Analyse. Funktioniert für jeden DataFrame.
    """
    uebersicht = pd.DataFrame(
        {
            "fehlend": df.isna().sum(),
            "anteil_%": (df.isna().mean() * 100).round(2),
            "vorhanden": df.notna().sum(),
        }
    )
    return uebersicht.sort_values("fehlend", ascending=False)


fehlende_werte(titanic)

Drei Spalten haben Lücken:

| Spalte | fehlend | Anteil | Einordnung nach der Strategiematrix der Vorlesung |
|---|---|---|---|
| `deck` | 688 | 77.2 % | über 20 %, die Spalte ist als Merkmal kaum brauchbar |
| `age` | 177 | 19.9 % | 5 bis 20 %, der interessante Fall |
| `embarked` | 2 | 0.2 % | unter 5 %, hier ist Zeilenlöschen unkritisch |

Damit ist die Zahl aus der Vorlesung belegt: 891 minus 714 ergibt 177.

### 3.2 Muster statt Anzahl

Jetzt der eigentliche Punkt. Der Anteil allein sagt nichts über den Mechanismus. Entscheidend ist,
**ob die Fehlrate von etwas abhängt, das wir beobachten können**. Das ist eine `groupby`-Rechnung.

In [ ]:
def fehlrate_nach_gruppe(df: pd.DataFrame, spalte: str, gruppe: str) -> pd.DataFrame:
    """Fehlrate einer Spalte, aufgeschlüsselt nach einer beobachteten Gruppenvariable."""
    fehlt = df[spalte].isna()
    nach_gruppe = fehlt.groupby(df[gruppe], observed=True)
    return pd.DataFrame(
        {
            "n": nach_gruppe.size(),
            "fehlend": nach_gruppe.sum(),
            "fehlrate_%": (nach_gruppe.mean() * 100).round(1),
        }
    )


print("Fehlrate von 'age' nach Passagierklasse:")
print(fehlrate_nach_gruppe(titanic, "age", "pclass"))

In [ ]:
print("Fehlrate von 'deck' nach Passagierklasse:")
print(fehlrate_nach_gruppe(titanic, "deck", "pclass"))

Das ist ein klares Muster. Bei `age` fehlt in der dritten Klasse mehr als viermal so oft wie in der
zweiten (27.7 % gegen 6.0 %). Bei `deck` ist es noch drastischer: 19.0 % gegen 97.6 %.

**Was folgt daraus?** Die Fehlrate hängt sichtbar von einer **beobachteten** Variable ab. Damit ist
reiner Zufall (MCAR) ausgeschlossen: Bei MCAR müssten die Fehlraten in allen Gruppen ungefähr gleich
sein. Und weil `pclass` im Datensatz steht, ist der Ausfall **modellierbar**, das ist die MAR-Logik
der Vorlesung.

Was hier ausdrücklich **nicht** folgt: dass es tatsächlich MAR ist. Dazu gleich mehr.

### 3.3 Die drei Mechanismen

An echten Daten lassen sich die Mechanismen grundsätzlich nicht nachweisen. Der Grund ist simpel:
Um zu prüfen, ob die fehlenden Alterswerte systematisch anders sind als die vorhandenen, bräuchten
wir die fehlenden Alterswerte. Genau die fehlen.

Deshalb drehen wir es um und erzeugen einen Datensatz, bei dem wir die Wahrheit kennen, und löschen
anschliessend selbst. Als Beispiel der Mini-Check aus der Vorlesung: **In einer Fitness-App
fehlen viele Gewichtseinträge.**

In [ ]:
# Ausgangslage: 5000 Nutzerinnen und Nutzer, deren wahres Gewicht wir kennen.
# Wer eine Smartwatch nutzt, wiegt im Schnitt etwas weniger als wer manuell eintraegt.
rng_daten = np.random.default_rng(SEED)
n = 5000

geraet = rng_daten.choice(["Smartwatch", "Manuell"], size=n, p=[0.6, 0.4])
gewicht_wahr = np.where(
    geraet == "Smartwatch",
    rng_daten.normal(72, 12, n),
    rng_daten.normal(82, 12, n),
).round(1)

fitness = pd.DataFrame({"geraet": geraet, "gewicht_wahr": gewicht_wahr})
WAHRER_MITTELWERT = fitness["gewicht_wahr"].mean()

print(f"Wahrer Mittelwert über alle {n} Personen: {WAHRER_MITTELWERT:.2f} kg\n")
print(fitness.groupby("geraet")["gewicht_wahr"].agg(["size", "mean"]).round(2))

In [ ]:
# Jetzt loeschen wir dreimal unterschiedlich, mit je rund 25 Prozent Ausfall.
rng_ausfall = np.random.default_rng(7)


def logistisch(x):
    return 1 / (1 + np.exp(-x))


# MCAR: Ausfallwahrscheinlichkeit fuer alle gleich.
#       Deck-Beispiel: der Log-Server droppt Pakete rein stochastisch.
p_mcar = np.full(n, 0.25)

# MAR:  Ausfall haengt am GERAET, also an einer beobachteten Spalte.
#       Wer manuell eintraegt, vergisst es oefter.
p_mar = np.where(geraet == "Manuell", 0.45, 0.11)

# MNAR: Ausfall haengt am GEWICHT selbst, also am fehlenden Wert.
#       Deck-Aufloesung des Mini-Checks: Menschen tragen ihr Gewicht nicht ein,
#       wenn es ihnen unangenehm ist.
p_mnar = logistisch((gewicht_wahr - 76) / 7) * 0.5

for name, p in [("MCAR", p_mcar), ("MAR", p_mar), ("MNAR", p_mnar)]:
    fehlt = rng_ausfall.random(n) < p
    fitness[f"gewicht_{name}"] = fitness["gewicht_wahr"].where(~fehlt)
    print(f"{name:5s} Fehlrate: {fehlt.mean():.1%}")

fitness.head()

Drei Spalten, jeweils rund ein Viertel fehlend. Nach Anteil und Anzahl sind sie **nicht zu
unterscheiden**. Jetzt die Frage, auf die es ankommt: Was macht das mit dem Mittelwert?

In [ ]:
def imputationsvergleich(df: pd.DataFrame, spalten: list[str], wahrheit: float) -> pd.DataFrame:
    """Stellt den wahren Mittelwert gegen Complete-Case und Median-Imputation."""
    zeilen = {}
    for spalte in spalten:
        s = df[spalte]
        complete_case = s.mean()                       # Listwise Deletion
        median_imput = s.fillna(s.median()).mean()     # einfache Imputation
        zeilen[spalte.replace("gewicht_", "")] = {
            "wahr": wahrheit,
            "complete_case": complete_case,
            "cc_fehler": complete_case - wahrheit,
            "median_imput": median_imput,
            "mi_fehler": median_imput - wahrheit,
        }
    return pd.DataFrame(zeilen).T


vergleich = imputationsvergleich(
    fitness, ["gewicht_MCAR", "gewicht_MAR", "gewicht_MNAR"], WAHRER_MITTELWERT
)
vergleich.round(2)

Das ist der Kern des ganzen Abschnitts:

- **MCAR:** Der Complete-Case-Mittelwert liegt 0.19 kg neben der Wahrheit, ein Viertelprozent.
  Genau das sagt die Vorlesung zu MCAR: Die Stichprobe schrumpft, die Schätzer bleiben unverfälscht.
  Listwise Deletion ist hier zulässig und kostet nur Präzision.
- **MAR:** Der Mittelwert liegt gut ein Kilogramm zu tief. Naives Weglassen erzeugt Bias.
- **MNAR:** Der Mittelwert liegt rund zweieinhalb Kilogramm zu tief, und die Median-Imputation macht
  es nicht besser, sondern minimal schlechter. Das ist die Warnung der Strategiematrix ausgerechnet:
  **Standard-Imputation erzeugt bei MNAR immer systematischen Bias.**

Die Richtung stimmt mit der Auflösung des Mini-Checks überein: Standard-Imputation **unterschätzt**
das wahre Durchschnittsgewicht, weil die fehlenden Werte systematisch am oberen Rand der Verteilung
liegen.

Bei MCAR bleibt trotzdem eine kleine Abweichung stehen. Ist das ein schwacher Bias oder blosses
Rauschen? Das ist genau die Unterscheidung aus Abschnitt 2, und hier lässt sie sich beantworten,
weil wir die Wahrheit kennen: Wir wiederholen denselben Ausfall mit fünf anderen Seeds.

In [ ]:
zeilen = []
for seed in [101, 102, 103, 104, 105]:
    zufall = np.random.default_rng(seed).random(n)
    eintrag = {"seed": seed}
    for name, p in [("MCAR", p_mcar), ("MAR", p_mar), ("MNAR", p_mnar)]:
        beobachtet = fitness["gewicht_wahr"].where(~(zufall < p))
        eintrag[name] = round(beobachtet.mean() - WAHRER_MITTELWERT, 2)
    zeilen.append(eintrag)

print("Fehler des Complete-Case-Mittelwerts, fünf Wiederholungen:")
print(pd.DataFrame(zeilen).set_index("seed"))

**MCAR schwankt um die Null und wechselt dabei das Vorzeichen. MAR und MNAR zeigen in jeder
einzelnen Wiederholung nach unten.** Das ist die Definition aus Abschnitt 2 als Tabelle: MCAR
erzeugt **Rauschen**, MAR und MNAR erzeugen einen **gerichteten Fehler**.

Und damit schliesst sich der Kreis zur Bias-Diskussion: Gegen die MCAR-Abweichung hilft eine
grössere Stichprobe, gegen die beiden anderen nicht.

### 3.4 Was sich aus den Daten erkennen lässt, und was nicht

Jetzt die Frage, die in der Praxis zählt: Hätten wir den Mechanismus erkennen können, ohne die
Wahrheit zu kennen? Dafür gibt es genau ein Werkzeug, die Fehlrate je beobachteter Gruppe.

In [ ]:
diagnose = pd.DataFrame(
    {
        name.replace("gewicht_", ""): fitness[name].isna().groupby(fitness["geraet"]).mean()
        for name in ["gewicht_MCAR", "gewicht_MAR", "gewicht_MNAR"]
    }
)
print("Fehlrate je Gerät (nur beobachtbare Information):")
print((diagnose * 100).round(1))

**MCAR ist widerlegbar.** Die Fehlraten sind in beiden Gruppen praktisch gleich (24.4 % gegen
25.2 %), das ist mit reinem Zufall vereinbar. Bei MAR (44.2 % gegen 11.5 %) und MNAR (31.2 % gegen
19.8 %) sind sie es nicht, dort ist MCAR damit ausgeschlossen.

**MAR und MNAR sind es nicht.** Beide zeigen unterschiedliche Fehlraten je Gerät. Aus dieser Tabelle
allein lässt sich nicht entscheiden, welcher der beiden Fälle vorliegt. Bei MNAR hängt der Ausfall
am Gewicht, und weil das Gewicht mit dem Gerät zusammenhängt, schlägt das auf die Gerätespalte durch.

Das ist keine Schwäche der Methode, sondern eine grundsätzliche Grenze: Der Unterschied zwischen MAR
und MNAR liegt darin, ob der Ausfall **zusätzlich** vom fehlenden Wert selbst abhängt, und dazu
bräuchte man den fehlenden Wert.

> **Die Entscheidung MAR gegen MNAR ist eine Annahme über die Erhebung, keine Rechnung.**
> Sie kommt aus dem Domänenwissen: Wie wurde gemessen, wann bricht der Vorgang ab, wer trägt nichts ein?

Dass das praktisch relevant ist, zeigt die nächste Rechnung. Die Strategiematrix der
Vorlesung empfiehlt bei MAR die **Gruppen-Imputation**, also den Median innerhalb der beobachteten
Gruppe statt über alle.

In [ ]:
def gruppen_imputation(df: pd.DataFrame, spalte: str, gruppe: str) -> pd.Series:
    """Füllt fehlende Werte mit dem Median der jeweiligen Gruppe (Strategiematrix, Spalte MAR)."""
    return df[spalte].fillna(df.groupby(gruppe, observed=True)[spalte].transform("median"))


ergebnis = {}
for name in ["gewicht_MCAR", "gewicht_MAR", "gewicht_MNAR"]:
    gefuellt = gruppen_imputation(fitness, name, "geraet")
    ergebnis[name.replace("gewicht_", "")] = {
        "wahr": WAHRER_MITTELWERT,
        "gruppen_imput": gefuellt.mean(),
        "fehler": gefuellt.mean() - WAHRER_MITTELWERT,
    }

print("Gruppen-Imputation (Median je Gerät):")
print(pd.DataFrame(ergebnis).T.round(2))

Die Gruppen-Imputation **repariert MAR** (Fehler von -1.02 auf -0.09) und **repariert MNAR nicht**
(Fehler von -2.42 auf -2.15). Das ist die Strategiematrix in Zahlen, und der Grund, warum die
Unterscheidung nicht akademisch ist: Dieselbe Methode ist im einen Fall richtig und im anderen
wirkungslos.

### 3.5 Strategien im Überblick

| Anteil | MCAR (Zufall) | MAR (Bedingt) | MNAR (Versteckt) |
| :--- | :--- | :--- | :--- |
| **< 5 %** | Zeilen löschen (Drop) | Gruppen-Imputation (Median/Modus) | Domain-Wissen nutzen |
| **5 bis 20 %** | Einfache Imputation / KNN | Multiple Imputation (MICE / KNN) | Selektionsmodelle (z. B. Heckman) |
| **> 20 %** | Multiple Imputation (MICE) | MICE + Sensitivitätsanalyse | Explizite Modellierung des Ausfalls |

In diesem Notebook sind die beiden linken Spalten codiert: Zeilen löschen (Complete Case),
einfache Imputation (Median) und Gruppen-Imputation. **MICE** (Multivariate Imputation by Chained
Equations), **k-NN-Imputation** und **Selektionsmodelle** brauchen zusätzliche Pakete
(`scikit-learn`, `statsmodels`) und kommen im Kurs später. Für den Moment reicht das Prinzip.

> **Take-Home Missing Values**
>
> 1. **Zuerst das Warum.** Der Mechanismus entscheidet, nicht der Anteil.
> 2. **Drei Typen:** MCAR (reiner Zufall), MAR (abhängig von Bekanntem), MNAR (abhängig vom fehlenden Wert).
> 3. **Nie blind imputieren.** Bei MNAR erzeugt jede Standard-Imputation systematischen Bias.

---

## 4. Erste EDA

> **Kernaussage der Vorlesung:** EDA heisst Muster erkennen, Fehler aufspüren und Hypothesen bilden. Und:
> **EDA erzeugt Hypothesen, sie bestätigt sie nicht.** Bestätigen lassen sie sich erst mit neuen
> Daten und einem Test, ab Woche 6.

Die Reihenfolge ist immer dieselbe: Struktur, dann Kennzahlen, dann Bilder.

### 4.1 Struktur

In [ ]:
print(f"Form des Datensatzes: {titanic.shape[0]} Zeilen, {titanic.shape[1]} Spalten\n")
titanic.head()

`head()` zeigt standardmässig die ersten fünf Zeilen. Schon hier sieht man die Spaltentypen, die
Wertebereiche und die ersten `NaN` in `deck`.

### 4.2 Kennzahlen

In [ ]:
titanic.describe()

Die Vorlesung nennt zwei Signale in dieser Tabelle. Beide lassen sich direkt nachrechnen:

In [ ]:
# Signal 1: count deckt fehlende Werte auf.
print("Signal 1, count gegen Zeilenzahl:")
for spalte in ["age", "fare"]:
    fehlend = titanic.shape[0] - titanic[spalte].count()
    print(f"  {spalte:5s} count = {titanic[spalte].count():3d} von {titanic.shape[0]} -> {fehlend} fehlen")

# Signal 2: Mittelwert gegen Median zeigt Schiefe.
print("\nSignal 2, Mittelwert gegen Median:")
for spalte in ["age", "fare"]:
    mw, med = titanic[spalte].mean(), titanic[spalte].median()
    print(f"  {spalte:5s} Mittelwert {mw:6.2f} | Median {med:6.2f} | Verhältnis {mw / med:.2f}")

print(f"\n  Maximum von fare: {titanic['fare'].max():.2f}")

Bei `age` liegen Mittelwert und Median dicht beieinander (29.70 gegen 28.00), die Verteilung ist
ungefähr symmetrisch. Bei `fare` ist der Mittelwert mehr als doppelt so gross wie der Median (32.20
gegen 14.45), und das Maximum liegt bei 512.33. Das ist eine **rechtsschiefe** Verteilung: wenige
sehr hohe Werte ziehen den Mittelwert nach oben, der Median bleibt davon unberührt.

Genau deshalb ist der Median bei schiefen Verteilungen das robustere Zentralmass. Die Details dazu
kommen in VL02.

`describe()` lässt standardmässig alles weg, was nicht numerisch ist. Die kategorialen Spalten
brauchen einen eigenen Aufruf:

In [ ]:
titanic.describe(exclude="number")

Andere Kennzahlen, weil andere Messniveaus: `count`, `unique`, `top` (der Modus) und `freq`. Ein
Mittelwert steht hier zu Recht nicht.

### 4.3 Bilder

Die Vorlesung zeigt drei Plots als Teaser. Hier laufen sie. Wichtig ist dabei ein Punkt, den die
Vorlesung nur nebenbei erwähnt: **stille Datenreduktion**. Sowohl
`hist()` als auch seaborn lassen Zeilen mit `NaN` kommentarlos weg. Deshalb steht vor jedem Plot,
wie viele Zeilen tatsächlich eingehen.

In [ ]:
verwendet = titanic["age"].notna().sum()
print(f"Histogramm über 'age': {verwendet} von {len(titanic)} Zeilen, "
      f"{len(titanic) - verwendet} fallen wegen NaN still weg.")

titanic["age"].hist(bins=20)
plt.title("Altersverteilung der Passagiere")
plt.xlabel("Alter in Jahren")
plt.ylabel("Anzahl Passagiere")
plt.show()

`bins=20` teilt den Wertebereich in 20 gleich breite Intervalle. Die Wahl ist nicht neutral: Zu
wenige Bins glätten Struktur weg, zu viele zeigen Rauschen als Struktur. VL03 geht darauf ein.

Ein Histogramm beschreibt **eine** metrische Variable. Für eine metrische Grösse über Gruppen nimmt
man den Boxplot.

In [ ]:
verwendet = titanic[["class", "fare"]].notna().all(axis=1).sum()
print(f"Boxplot: {verwendet} von {len(titanic)} Zeilen, keine fallen weg.")

sns.boxplot(x="class", y="fare", data=titanic)
plt.title("Ticketpreis nach Passagierklasse")
plt.xlabel("Klasse")
plt.ylabel("Ticketpreis")
plt.show()

Die Punkte oberhalb der Whisker sind nach der Standardregel **Ausreisser**. Was genau Box, Whisker
und diese Regel bedeuten, kommt in VL02 und VL03. Hier reicht: Die erste Klasse streut massiv
stärker als die dritte, und die Extremwerte, die den Mittelwert nach oben ziehen, sind sichtbar.

Für den Zusammenhang zweier metrischer Variablen der Scatterplot.

In [ ]:
verwendet = titanic[["age", "fare"]].notna().all(axis=1).sum()
print(f"Scatterplot: {verwendet} von {len(titanic)} Zeilen, "
      f"{len(titanic) - verwendet} fallen wegen fehlender Altersangaben weg.")

sns.scatterplot(x="age", y="fare", data=titanic, alpha=0.6)
plt.title("Ticketpreis gegen Alter")
plt.xlabel("Alter in Jahren")
plt.ylabel("Ticketpreis")
plt.show()

Fast jede fünfte Zeile fehlt in diesem Bild, ohne dass es irgendwo steht. Wer daraus eine Aussage
über "die Passagiere" ableitet, meint in Wahrheit "die Passagiere mit bekanntem Alter", und die sind
nach Abschnitt 3.2 systematisch anders verteilt als der Rest.

Ob zwischen Alter und Ticketpreis ein Zusammenhang besteht und wie man ihn misst, ist Thema von VL04.

### 4.4 Kategoriale Variablen

Das Histogramm hat ein Gegenstück für nominale und ordinale Daten: die Häufigkeitstabelle,
beziehungsweise der Balken-Plot.

In [ ]:
for spalte in ["class", "embarked", "who"]:
    print(f"--- {spalte} ---")
    print(titanic[spalte].value_counts(dropna=False))
    print()

In [ ]:
fig, achsen = plt.subplots(1, 2, figsize=(11, 3.8))

sns.countplot(x="class", data=titanic, ax=achsen[0],
              order=["First", "Second", "Third"])
achsen[0].set_title("Passagiere je Klasse")
achsen[0].set_xlabel("Klasse")
achsen[0].set_ylabel("Anzahl")

sns.countplot(x="embarked", data=titanic, ax=achsen[1])
achsen[1].set_title("Zustiegshafen")
achsen[1].set_xlabel("Hafen")
achsen[1].set_ylabel("Anzahl")

plt.tight_layout()
plt.show()

> **Was bewusst noch nicht hier steht.** KDE, ECDF, QQ-Plot, die Ausreisserregel und die Parameter
> von `boxplot` und `violinplot` gehören zu VL02 und VL03. VL01 ist die erste Inspektion, nicht die
> Verteilungsdiagnostik.

---

## 5. Zusammenfassung und Übertragung

Die vier Lernziele der Vorlesung, jeweils mit der Stelle, an der sie in diesem Notebook laufen:

| # | Lernziel | Abschnitt |
|---|---|---|
| 01 | **Datentypen** unterscheiden und das Messniveau bestimmen | 1 |
| 02 | **Bias** erkennen und benennen | 2 |
| 03 | **Missing Values** diagnostizieren und den Mechanismus einordnen | 3 |
| 04 | **Erste EDA** durchführen | 4 |

Die wichtigsten drei Sätze des Notebooks:

1. Der **dtype sagt nichts über das Messniveau**. Die Zuordnung ist eine inhaltliche Entscheidung.
2. **Mehr Daten beseitigen keinen Bias.** Sie schätzen ihn präziser.
3. Bei fehlenden Werten entscheidet der **Mechanismus**, nicht der Anteil. Und ob MAR oder MNAR
   vorliegt, ist eine Annahme über die Erhebung, keine Rechnung.

### Was ihr mit eurem eigenen Datensatz macht

Die Reihenfolge ist nicht beliebig. Wer sie umdreht und mit dem Plot anfängt, produziert Bilder,
deren Grundlage er nicht kennt.

**1. Überblick, bevor die erste Kennzahl fällt.** Wie viele Zeilen, wie viele Spalten, was steht
pro Spalte drin, wie viele verschiedene Werte gibt es.

**2. Für jede Spalte das Messniveau notieren.** Nicht den dtype, das Messniveau. Diese Zuordnung
nimmt euch kein Programm ab, und sie entscheidet, welche Kennzahl überhaupt zulässig ist. Konkret:
Bei nominalen und ordinalen Spalten berichtet ihr keinen Mittelwert, auch wenn dort `int64` steht
und pandas ihn bereitwillig ausrechnet.

**3. Fragen, wie die Daten entstanden sind.** Wer ist in den Daten und wer nicht, welcher Filter
lag vor der Erfassung, wie wurde gemessen, und gibt es Werte, die es gar nicht geben kann. Das sind
die vier Fragen aus Abschnitt 2. Beantworten lassen sie sich nicht aus der Tabelle, sondern nur aus
der Dokumentation des Datensatzes oder durch Nachfragen bei der Stelle, die ihn erhoben hat.

**4. Bei den Lücken das Muster ansehen, nicht die Anzahl.** „12 Prozent fehlen" ist keine Diagnose.
Die Diagnose ist die Fehlrate je Gruppe. Variiert sie zwischen den Gruppen, ist MCAR erledigt und
naives Zeilenlöschen erzeugt Bias. Ob danach MAR oder MNAR vorliegt, entscheidet keine Rechnung,
sondern euer Wissen darüber, wie erhoben wurde.

**5. Erst jetzt entscheiden, was mit den Lücken passiert.** Löschen, imputieren oder die Spalte
fallen lassen, je nach Mechanismus und Anteil. Und die Entscheidung hinschreiben, mit Begründung.

**6. Zum Schluss die Bilder.** Bei jedem Plot einmal nachsehen, wie viele Zeilen wegen `NaN` still
wegfallen. Eine Aussage über „alle Nutzer" auf Basis eines Plots, in dem ein Fünftel fehlt, ist
eine Aussage über vier Fünftel der Nutzer.

**Was ihr festhalten solltet.** Nicht die Ausgaben, die entstehen jederzeit neu, sondern die
**Entscheidungen**: welches Messniveau ihr je Spalte angesetzt habt, wie ihr mit den Lücken
umgegangen seid und warum, und welche Verzerrung ihr in den Daten vermutet. Das ist der Teil, den
ihr später verteidigen müsst, und der Teil, den ihr in drei Wochen selbst nicht mehr
rekonstruieren könnt.

**Was VL01 noch nicht beantwortet.** Ob zwischen zwei Variablen ein Zusammenhang besteht, ist VL04.
Ob ein beobachteter Unterschied belastbar ist oder Zufall, beginnt in VL06. Alles, was ihr in
dieser ersten Runde findet, ist eine **Hypothese**. Die EDA erzeugt sie, bestätigen kann sie sie
nicht.

**Fürs Projekt.** Lauft diese sechs Schritte einmal durch, **bevor** ihr euch auf einen Datensatz
festlegt. Ein Datensatz, bei dem schon Schritt 2 unklar bleibt oder bei dem Schritt 4 ein Muster
zeigt, das niemand erklären kann, kostet euch später Wochen.

---

**Weiter in VL02:** Lage, Streuung und Verteilungsform, also Mittelwert gegen Median, Quantile,
Varianz, Standardabweichung und robuste Kennzahlen. Die Schiefe von `fare`, die in Abschnitt 4.2
aufgetaucht ist, wird dort zum Thema.